In [ ]:
# Cell 1: imports and project-root setup
import sys
from pathlib import Path

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    candidates = [start] + list(start.parents)
    for candidate in candidates:
        if (candidate / 'src').exists():
            return candidate
    return start

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
import gym
import d4rl  # noqa: F401

from src.experiment_config import *
from src.config import CHECKPOINTS_DIR, RAW_METRICS_DIR, OBS_STATS_DIR
from src.dataset import NoisyOfflineRLDataset
from src.denoised_mdp_encoder import DenoisedMDPEncoder
from src.iql import IQLAgent
from src.train_eval import (
    eval_policy_on_env,
    load_and_freeze_encoder,
    train_iql_from_loader,
    save_metrics_json,
)

In [ ]:
# Cell 2: experiment configuration

METHOD = 'denoised_mdp'

def noise_tag(noise_dim, noise_scale, noise_type):
    ns = str(noise_scale).replace('.', 'p')
    return f'nd{noise_dim}_ns{ns}_{noise_type}'

NOISE_TAG = noise_tag(NOISE_DIM, NOISE_SCALE, NOISE_TYPE)
SEED_TAG = f'seed_{SEED}'

CKPT_DIR    = CHECKPOINTS_DIR / METHOD / ENV_NAME / NOISE_TAG / SEED_TAG
METRICS_DIR = RAW_METRICS_DIR / METHOD / ENV_NAME / NOISE_TAG / SEED_TAG
OBS_DIR     = OBS_STATS_DIR   / METHOD / ENV_NAME / NOISE_TAG / SEED_TAG

for d in [CKPT_DIR, METRICS_DIR, OBS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('DEVICE:', DEVICE)
print('METHOD:', METHOD)
print('CKPT_DIR:', CKPT_DIR)
print('METRICS_DIR:', METRICS_DIR)

In [ ]:
# Cell 3: dataset and dataloader

dataset = NoisyOfflineRLDataset(
    env_name=ENV_NAME,
    noise_dim=NOISE_DIM,
    noise_scale=NOISE_SCALE,
    seed=SEED,
    use_timeouts=True,
    noise_type=NOISE_TYPE,
)

train_loader = DataLoader(
    dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
    num_workers=4, pin_memory=True, persistent_workers=True,
)

state_dim      = dataset.noisy_obs.shape[1]
action_dim     = dataset.actions.shape[1]
true_state_dim = dataset.obs_dim
LATENT_DIM     = int(max(true_state_dim, NOISE_DIM) * 1.5)

np.savez(
    OBS_DIR / 'obs_stats.npz',
    obs_mean=dataset.obs_mean,
    obs_std=dataset.obs_std,
    true_state_dim=true_state_dim,
)
print('Saved obs stats:', OBS_DIR / 'obs_stats.npz')
print('state_dim (noisy):', state_dim)
print('true_state_dim:', true_state_dim)
print('action_dim:', action_dim)
print('latent_dim:', LATENT_DIM)

In [ ]:
# Cell 4: helper — cross-covariance independence penalty

def cross_covariance_loss(z1, z2):
    """Push cross-covariance between z_task and z_irrel toward zero."""
    z1_norm = (z1 - z1.mean(dim=0)) / (z1.std(dim=0) + 1e-5)
    z2_norm = (z2 - z2.mean(dim=0)) / (z2.std(dim=0) + 1e-5)
    c = torch.mm(z1_norm.T, z2_norm) / z1.size(0)
    return (c ** 2).mean()

In [ ]:
# Cell 5: pretrain DenoisedMDPEncoder
#
# Loss = L_task_dyn + L_irrel_dyn + L_reward + λ * L_indep
#
# L_task_dyn:  task_dynamics(z_task(obs), action)  ≈  z_task(next_obs)  [stop-grad target]
# L_irrel_dyn: irrel_dynamics(z_irrel(obs))         ≈  z_irrel(next_obs) [stop-grad target]
# L_reward:    reward_predictor(z_task(obs))         ≈  reward
# L_indep:     cross_covariance(z_task, z_irrel)
#
# Key: NO pure_next_obs used — fully self-supervised, no privileged information.

torch.manual_seed(SEED)
np.random.seed(SEED)

encoder = DenoisedMDPEncoder(
    state_dim=state_dim,
    action_dim=action_dim,
    latent_dim=LATENT_DIM,
).to(DEVICE)

optimizer = torch.optim.Adam(encoder.parameters(), lr=3e-4)

pretrain_loader = DataLoader(
    dataset, batch_size=PRETRAIN_BS, shuffle=True, drop_last=True,
    num_workers=4, pin_memory=True, persistent_workers=True,
)

INDEP_WEIGHT = 50.0

for epoch in range(1, PRETRAIN_EPOCHS + 1):
    encoder.train()
    losses = []

    for obs, act, next_obs, rew, done, _pure_obs, _pure_next_obs in pretrain_loader:
        obs      = obs.to(DEVICE)
        act      = act.to(DEVICE)
        next_obs = next_obs.to(DEVICE)
        rew      = rew.to(DEVICE)

        z_task, z_irrel = encoder(obs)

        # Stop-gradient targets from the next noisy observation.
        with torch.no_grad():
            z_task_next_target, z_irrel_next_target = encoder(next_obs)

        # L_task_dyn: task branch must predict next z_task given action.
        z_task_pred = encoder.task_dynamics(torch.cat([z_task, act], dim=-1))
        loss_task_dyn = F.mse_loss(z_task_pred, z_task_next_target)

        # L_irrel_dyn: irrel branch must predict next z_irrel WITHOUT action.
        z_irrel_pred = encoder.irrel_dynamics(z_irrel)
        loss_irrel_dyn = F.mse_loss(z_irrel_pred, z_irrel_next_target)

        # L_reward: task branch predicts reward.
        pred_rew = encoder.reward_predictor(z_task)
        loss_rew = F.mse_loss(pred_rew.squeeze(-1), rew)

        # L_indep: decorrelate task and irrel latents.
        loss_indep = cross_covariance_loss(z_task, z_irrel)

        loss = loss_task_dyn + loss_irrel_dyn + loss_rew + INDEP_WEIGHT * loss_indep

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(encoder.parameters(), max_norm=1.0)
        optimizer.step()
        losses.append(loss.detach())

    mean_loss = torch.stack(losses).mean().item()
    print(f'[pretrain][{METHOD}] epoch {epoch}, loss={mean_loss:.4f}')

CKPT_ENCODER = CKPT_DIR / f'encoder_epoch_{PRETRAIN_EPOCHS}.pth'
torch.save(encoder.state_dict(), CKPT_ENCODER)
print('Saved encoder:', CKPT_ENCODER)

In [ ]:
# Cell 6: freeze encoder and train IQL

encoder = load_and_freeze_encoder(
    encoder=encoder,
    ckpt_path=CKPT_ENCODER,
    device=DEVICE,
)

iql = IQLAgent(
    latent_dim=LATENT_DIM,
    action_dim=action_dim,
    device=DEVICE,
    expectile=0.7,
    temperature=3.0,
    discount=0.99,
)

iql_history = train_iql_from_loader(
    iql=iql,
    train_loader=train_loader,
    device=DEVICE,
    epochs=EPOCHS,
    ckpt_dir=CKPT_DIR,
    method=METHOD,
    save_every=10,
    encoder=encoder,
    repr_mode='denoised_mdp',
    use_tqdm=False,
)

In [ ]:
# Cell 7: evaluate and save metrics

print('Start evaluating ...')
metrics = eval_policy_on_env(
    iql=iql,
    env_name=ENV_NAME,
    encoder=encoder,
    method=METHOD,
    obs_mean=dataset.obs_mean,
    obs_std=dataset.obs_std,
    true_state_dim=true_state_dim,
    noise_dim=NOISE_DIM,
    noise_scale=NOISE_SCALE,
    noise_type=NOISE_TYPE,
    episodes=20,
    max_steps=1000,
    seed=SEED,
    device=DEVICE,
    use_fixed_noise=True,
)

metrics_path = save_metrics_json(
    metrics_dir=METRICS_DIR,
    metrics=metrics,
    env_name=ENV_NAME,
    method=METHOD,
    seed=SEED,
    noise_dim=NOISE_DIM,
    noise_scale=NOISE_SCALE,
    noise_type=NOISE_TYPE,
    extra={
        'latent_dim': LATENT_DIM,
        'pretrain_epochs': PRETRAIN_EPOCHS,
        'iql_epochs': EPOCHS,
        'indep_weight': INDEP_WEIGHT,
        'encoder_checkpoint': str(CKPT_ENCODER),
        'iql_history': iql_history,
    },
)

print('Saved metrics:', metrics_path)
print(metrics)